# conv-kernel-shape — worked example 3: Compare param counts: depthwise vs standard Conv2d

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-kernel-shape`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A **depthwise** conv sets `groups = in_channels`, so each input channel gets its own filter and the weight shape becomes `(OC, IC/groups, KH, KW) = (IC, 1, KH, KW)` (with OC=IC). A **standard** conv (groups=1) has weight shape `(OC, IC, KH, KW)`. Reading `weight.shape[1]` directly tells you `IC/groups`, the per-group input channels.

## Worked solution

**Step 1 — standard conv.** `nn.Conv2d(8, 8, 3)` has groups=1, so weight shape is `(8, 8, 3, 3)` and param count is `8*8*3*3 = 576`.

**Step 2 — depthwise conv.** `nn.Conv2d(8, 8, 3, groups=8)` splits the 8 input channels into 8 groups of 1. The weight's second axis collapses to `IC/groups = 1`, giving shape `(8, 1, 3, 3)` and only `8*1*3*3 = 72` params.

**Step 3 — read the layout, not the constructor.** We compute each count straight from `weight.shape` (`OC * (IC/groups) * KH * KW = weight.numel()`), so the function never needs to know `groups` separately — the second axis already encodes it.

**Step 4 — the ratio.** Depthwise uses `1/IC` of the per-spatial parameters of a same-channel standard conv: `72 / 576 = 1/8`. That 8x reduction is exactly the `groups=8` factor, which is why depthwise separable convs are so cheap.

In [ ]:
def conv_param_count(conv) -> int:
    return int(conv.weight.numel())

t.manual_seed(0)
std = t.nn.Conv2d(8, 8, kernel_size=3)
depth = t.nn.Conv2d(8, 8, kernel_size=3, groups=8)
n_std = conv_param_count(std)
n_depth = conv_param_count(depth)
print('standard', tuple(std.weight.shape), n_std)
print('depthwise', tuple(depth.weight.shape), n_depth)
print('ratio', n_depth / n_std)